In [2]:
!nvidia-smi

Mon Apr 13 13:38:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 16.6 MB/s eta 0:00:00


In [4]:
!pip install --upgrade accelerate
!pip unistall -y transformers accelerate
!pip install transformers==5.5.3

ERROR: unknown command "unistall" - maybe you meant "uninstall"
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [5]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
#From datasets import load_dataset, load_metric

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt") # Para tokenizar sentences

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

### Funcionalidad básica de modelo de HuggingFace

In [6]:
from transformers import AutoTokenizer, PegasusForConditionalGeneration

model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")

ARTICLE_TO_SUMMARIZE = (
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds "
    "amid dry conditions. The aim is to reduce the risk of wildfires. Nearly 800 thousand customers were "
    "scheduled to be affected by the shutoffs which were expected to last through at least midday tomorrow."
)
inputs = tokenizer(ARTICLE_TO_SUMMARIZE, max_length=1024, return_tensors="pt")

# Generate Summary
summary_ids = model.generate(inputs["input_ids"])
tokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

"California's largest electricity provider has turned off power to hundreds of thousands of customers."

### FINE TUNING

In [7]:
from transformers import AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
#Cargamos el modelo y lo tokenizamos
model = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model).to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [9]:
#Descargamos  y descomprimimos los datos. Se guardarán en la carpeta content.
!wget -c https://huggingface.co/datasets/knkarthick/samsum/resolve/main/train.csv
!wget -c https://huggingface.co/datasets/knkarthick/samsum/resolve/main/test.csv
!wget -c https://huggingface.co/datasets/knkarthick/samsum/resolve/main/validation.csv

--2026-04-13 13:42:33--  https://huggingface.co/datasets/knkarthick/samsum/resolve/main/train.csv
Resolving huggingface.co (huggingface.co)... 3.163.189.74, 3.163.189.114, 3.163.189.90, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.74|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/knkarthick/samsum/6b929ff10edec703164e3ddb2e94aae058c9ab5f/train.csv?%2Fdatasets%2Fknkarthick%2Fsamsum%2Fresolve%2Fmain%2Ftrain.csv=&etag=%22d9a64f87d3ffe64830175a611714510c405e350a%22 [following]
--2026-04-13 13:42:33--  https://huggingface.co/api/resolve-cache/datasets/knkarthick/samsum/6b929ff10edec703164e3ddb2e94aae058c9ab5f/train.csv?%2Fdatasets%2Fknkarthick%2Fsamsum%2Fresolve%2Fmain%2Ftrain.csv=&etag=%22d9a64f87d3ffe64830175a611714510c405e350a%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 9255369 (8.8M) [text/plain]
Saving to: ‘train.csv’

train.csv       

In [10]:

dataset = load_dataset(
    "csv",
    data_files={
        "train": "train.csv",
        "validation": "validation.csv",
        "test": "test.csv"
    }
)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [11]:

def convert_examples_to_features(example_batch):

    model_inputs = tokenizer(
        example_batch["dialogue"],
        max_length=1024,
        truncation=True,
        padding=True
    )

    labels = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True,
        padding=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs



In [12]:
dataset_samsun_pt = dataset.map(convert_examples_to_features, batched = True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [13]:
dataset_samsun_pt

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [14]:
# ENTRENAMIENTO

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model = model_pegasus)

In [15]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir = 'pegasus-samsum', num_train_epochs = 1, warmup_steps = 500,
    per_device_train_batch_size = 1, per_device_eval_batch_size = 1,
    weight_decay = 0.01, logging_steps = 10,
    eval_strategy='steps', eval_steps = 500, save_steps = int(1e6),
    gradient_accumulation_steps = 16
)

In [16]:
trainer = Trainer(model = model_pegasus, args = trainer_args,
                  processing_class = tokenizer, data_collator = seq2seq_data_collator,
                  train_dataset = dataset_samsun_pt["train"].select(range(1000)),
                  eval_dataset = dataset_samsun_pt['validation'])


In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
63,141.822437,8.483583


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=144.6502205984933, metrics={'train_runtime': 719.272, 'train_samples_per_second': 1.39, 'train_steps_per_second': 0.088, 'total_flos': 1681758584832000.0, 'train_loss': 144.6502205984933, 'epoch': 1.0})

# Evaluation

In [24]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Divide el dataset en fragmentos más pequeños para procesarlos en batches."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device="cuda",
                                column_text="dialogue",
                                column_summary="summary"):

    # Dividir en batches
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    model.to(device)
    model.eval()

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):

        # Tokenizar input
        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        ).to(device)

        # ✅ CORRECCIÓN AQUÍ (usar inputs reales)
        summaries = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        # Decodificar resúmenes generados
        decoded_summaries = [
            tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for s in summaries
        ]

        # Limpiar espacios extra
        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        # Añadir resultados a la métrica
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    return metric.compute()

In [19]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [26]:
import evaluate

rouge_metric = evaluate.load('rouge')
rouge_names = ["rouge1", "rouge2","rougeL","rougeLsum"]

In [27]:
rouge_metric

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
        Valid names:
        `"rouge{n}"` (e.g. `"rouge1"`, `"rouge2"`) where: {n} is the n-gram based scoring,
        `"rougeL"`: Longest common subsequence based scoring.
        `"rougeLsum"`: rougeLsum splits text using `"
"`.
        See details in https://github.com/huggingface/datasets/issues/617
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
 

In [28]:
score = calculate_metric_on_test_ds(dataset_samsun_pt['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text='dialogue',
column_summary = 'summary')

rouge_dict = {rn: score[rn] for rn in rouge_names}

import pandas as pd
pd.DataFrame(rouge_dict, index=[f'pegasus'])

100%|██████████| 5/5 [00:27<00:00,  5.55s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.019758,0.0,0.01969,0.019564


## Interpretación de resultados positivos o negativos

✅ ROUGE ≈ 1.0 — Excelente

Hay un fuerte solapamiento entre el resumen generado y el real.
El modelo captó casi todo el contenido importante.
Es muy positivo.

✅ ROUGE entre 0.7 y 0.9 — Bueno

Buen nivel de coincidencia.
El modelo entiende bien la tarea.
Es positivo, útil y coherente.

✅ ROUGE entre 0.5 y 0.7 — Regular

Coincidencia moderada.
El modelo captura parte del significado, pero pierde detalles.
Es aceptable, pero necesita mejorar.

❌ ROUGE < 0.5 — Malo

Muy poco solapamiento.
El modelo no resume bien, o no entiende el contexto.
Es negativo.

In [29]:
## Guardamos el modelo
model_pegasus.save_pretrained("pegasus-samsun-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [32]:
## Guardamos Tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [46]:
# Cargamos el tokenizer
model = PegasusForConditionalGeneration.from_pretrained("pegasus-samsun-model").to(device)
tokenizer = AutoTokenizer.from_pretrained("tokenizer")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

In [47]:
sample_text = dataset_samsun_pt["test"][0]['dialogue']

reference = dataset_samsun_pt["test"][0]['summary']

# Tokenizar y generar
inputs = tokenizer(sample_text, max_length=1024, truncation=True, return_tensors="pt").to(device)
summary_ids = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    length_penalty=0.8,
    num_beams=8,
    max_length=128
)

# Decodificar
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
print("Diálogo:")
print(sample_text)
print("\nResumen:")
print(summary)

Diálogo:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Resumen:
Amanda: Ask Larry Amanda: He called her last time we were at the park together.<n>Hannah: I'd rather you texted him.<n>Amanda: Just text him.
